# Telco Customer Churn: XGBoost + SHAP & LIME Explainability

**Goal**: Build a churn prediction model, then use SHAP and LIME to explain *why* the model makes its predictions.

**Pipeline**: Data Prep → XGBoost Model → SHAP (Global + Local) → LIME (Local) → Interpretation

## Section 1: Import Required Libraries

In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score,
                             ConfusionMatrixDisplay, classification_report)
from xgboost import XGBClassifier
import shap
import lime
import lime.lime_tabular

print('All packages imported successfully!')
print(f'  pandas  {pd.__version__}')
print(f'  numpy   {np.__version__}')
print(f'  shap    {shap.__version__}')


All packages imported successfully!
  pandas  3.0.1
  numpy   2.4.3
  shap    0.51.0


## Section 2: Load and Prepare Dataset

In [2]:
DATA_PATH = '/Users/hari.durai/Documents/AIApprenticeship/Explainability_Bias_Reporting_24March2026/handson-bias-reporting/WA_Fn-UseC_-Telco-Customer-Churn.csv'
WORKSPACE = '/Users/hari.durai/Documents/AIApprenticeship/Explainability_Bias_Reporting_24March2026/handson-bias-reporting'

df = pd.read_csv(DATA_PATH)
print(f'Raw shape: {df.shape}')
print(f'\nChurn distribution:\n{df["Churn"].value_counts()}')
print(f'\nChurn rate: {df["Churn"].value_counts(normalize=True)["Yes"]:.1%}')
print(f'\nSample data types:\n{df.dtypes.head(10)}')

Raw shape: (7043, 21)

Churn distribution:
Churn
No     5174
Yes    1869
Name: count, dtype: int64

Churn rate: 26.5%

Sample data types:
customerID           str
gender               str
SeniorCitizen      int64
Partner              str
Dependents           str
tenure             int64
PhoneService         str
MultipleLines        str
InternetService      str
OnlineSecurity       str
dtype: object


In [3]:
# 1. Drop non-predictive ID
df.drop('customerID', axis=1, inplace=True)

# 2. Fix TotalCharges: stored as string, blank for tenure=0 customers
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
print(f'TotalCharges NaN count: {df["TotalCharges"].isna().sum()}')
df['TotalCharges'].fillna(0, inplace=True)

# 3. Encode target
df['Churn'] = (df['Churn'] == 'Yes').astype(int)

# 4. Label-encode binary categoricals
binary_cols = ['gender', 'Partner', 'Dependents', 'PhoneService', 'PaperlessBilling']
le = LabelEncoder()
for col in binary_cols:
    df[col] = le.fit_transform(df[col])

# 5. One-hot encode multi-class categoricals
multiclass_cols = [
    'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup',
    'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies',
    'Contract', 'PaymentMethod'
]
df = pd.get_dummies(df, columns=multiclass_cols, drop_first=False)
bool_cols = df.select_dtypes(include='bool').columns
df[bool_cols] = df[bool_cols].astype(int)

print(f'\nFinal shape after encoding: {df.shape}')
print(f'Feature count: {df.shape[1] - 1}')

TotalCharges NaN count: 11

Final shape after encoding: (7043, 41)
Feature count: 40


In [4]:
X = df.drop('Churn', axis=1)
y = df['Churn']
feature_names = list(X.columns)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Train: {X_train.shape}  |  Test: {X_test.shape}')
print(f'Train churn rate: {y_train.mean():.1%}  |  Test churn rate: {y_test.mean():.1%}')

numeric_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']
scaler = StandardScaler()
X_train = X_train.copy()
X_test  = X_test.copy()
X_train[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
X_test[numeric_cols]  = scaler.transform(X_test[numeric_cols])
print(f'\nScaling applied to: {numeric_cols}')
print(f'tenure mean after scaling: {X_train["tenure"].mean():.4f} (should be ~0)')

Train: (5634, 40)  |  Test: (1409, 40)
Train churn rate: 26.5%  |  Test churn rate: 26.5%

Scaling applied to: ['tenure', 'MonthlyCharges', 'TotalCharges']
tenure mean after scaling: -0.0000 (should be ~0)


## Section 3: Train a Machine Learning Model (XGBoost)

**Why XGBoost?** Best performance on tabular data, exact SHAP support via `TreeExplainer`, handles class imbalance via `scale_pos_weight`.

In [5]:
scale_pos_weight = y_train.value_counts()[0] / y_train.value_counts()[1]
print(f'scale_pos_weight (non-churn / churn): {scale_pos_weight:.2f}')

model = XGBClassifier(
    n_estimators=200,
    max_depth=5,
    learning_rate=0.1,
    scale_pos_weight=scale_pos_weight,
    eval_metric='logloss',
    random_state=42,
    n_jobs=-1
)
model.fit(X_train, y_train)
print('Model training complete!')

scale_pos_weight (non-churn / churn): 2.77


Model training complete!


In [6]:
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print('=' * 55)
print('  MODEL EVALUATION ON TEST SET')
print('=' * 55)
print(f'  Accuracy:  {accuracy_score(y_test, y_pred):.4f}')
print(f'  Precision: {precision_score(y_test, y_pred):.4f}')
print(f'  Recall:    {recall_score(y_test, y_pred):.4f}')
print(f'  F1-Score:  {f1_score(y_test, y_pred):.4f}')
print(f'  AUC-ROC:   {roc_auc_score(y_test, y_prob):.4f}')
print(f'\n{classification_report(y_test, y_pred, target_names=["No Churn", "Churn"])}')

fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred, display_labels=['No Churn', 'Churn'], ax=ax, cmap='Blues'
)
ax.set_title('Confusion Matrix - XGBoost', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{WORKSPACE}/confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.close()
print('Saved: confusion_matrix.png')

  MODEL EVALUATION ON TEST SET
  Accuracy:  0.7495
  Precision: 0.5196
  Recall:    0.7433
  F1-Score:  0.6117
  AUC-ROC:   0.8282

              precision    recall  f1-score   support

    No Churn       0.89      0.75      0.82      1035
       Churn       0.52      0.74      0.61       374

    accuracy                           0.75      1409
   macro avg       0.70      0.75      0.71      1409
weighted avg       0.79      0.75      0.76      1409

Saved: confusion_matrix.png


## Section 4: SHAP — Compute SHAP Values

**What is SHAP?** Each feature gets a 'credit' for the prediction, computed using game theory. `TreeExplainer` gives exact Shapley values for tree models — no approximation needed.

In [7]:
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)

print(f'SHAP values shape: {shap_values.shape}')
print(f'  -> one row per test sample, one column per feature')
print(f'\nBase (expected) value: {explainer.expected_value:.4f}')
print(f'  -> average model margin output across all training data')

# Sanity check: SHAP sum + base = model raw margin output
sample_idx = 0
shap_sum = shap_values[sample_idx].sum() + explainer.expected_value
model_margin = model.predict(X_test.iloc[[sample_idx]], output_margin=True)[0]
print(f'\nVerification for test sample 0:')
print(f'  SHAP sum + base = {shap_sum:.4f}')
print(f'  Model margin    = {model_margin:.4f}')
print(f'  Match: {abs(shap_sum - model_margin) < 0.01}')

SHAP values shape: (1409, 40)
  -> one row per test sample, one column per feature

Base (expected) value: -0.0091
  -> average model margin output across all training data

Verification for test sample 0:
  SHAP sum + base = -5.5234
  Model margin    = -5.5234
  Match: True


## Section 5: SHAP — Summary Plot (Global Feature Importance)

**Beeswarm plot**: each dot = one customer. Red = high feature value, Blue = low. X-axis = SHAP value (right = pushes toward churn).

In [8]:
# Beeswarm
plt.figure(figsize=(12, 8))
shap.summary_plot(
    shap_values, X_test, feature_names=feature_names,
    plot_type='dot', max_display=20, show=False
)
plt.title('SHAP Summary Plot (Beeswarm) - Top 20 Features', fontsize=13, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig(f'{WORKSPACE}/shap_summary_beeswarm.png', dpi=150, bbox_inches='tight')
plt.close()
print('Saved: shap_summary_beeswarm.png')

# Bar chart of mean |SHAP|
plt.figure(figsize=(10, 7))
shap.summary_plot(
    shap_values, X_test, feature_names=feature_names,
    plot_type='bar', max_display=15, show=False
)
plt.title('SHAP Feature Importance (Mean |SHAP Value|)', fontsize=13, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig(f'{WORKSPACE}/shap_bar.png', dpi=150, bbox_inches='tight')
plt.close()
print('Saved: shap_bar.png')

# Print top 10 ranked features
mean_abs_shap = np.abs(shap_values).mean(axis=0)
top10_idx = np.argsort(mean_abs_shap)[::-1][:10]
print(f'\nTop 10 Global Churn Drivers (SHAP):')
print(f'{"Rank":<5} {"Feature":<50} {"Mean |SHAP|":>12}')
print('-' * 70)
for rank, idx in enumerate(top10_idx, 1):
    print(f'{rank:<5} {feature_names[idx]:<50} {mean_abs_shap[idx]:>12.4f}')

Saved: shap_summary_beeswarm.png
Saved: shap_bar.png

Top 10 Global Churn Drivers (SHAP):
Rank  Feature                                             Mean |SHAP|
----------------------------------------------------------------------
1     Contract_Month-to-month                                  0.8254
2     tenure                                                   0.6534
3     MonthlyCharges                                           0.4270
4     TotalCharges                                             0.3297
5     OnlineSecurity_No                                        0.2764
6     TechSupport_No                                           0.1908
7     PaymentMethod_Electronic check                           0.1856
8     InternetService_Fiber optic                              0.1814
9     Contract_Two year                                        0.1593
10    MultipleLines_No                                         0.1333


## Section 6: SHAP — Waterfall Plot (Force Plot for Individual Predictions)

A **waterfall plot** explains one customer's prediction step by step. Red bars push toward churn, blue bars push away from churn.

In [9]:
# Pick representative customers
y_test_arr = np.array(y_test)
y_pred_arr = np.array(y_pred)

churner_idx     = int(np.where((y_test_arr == 1) & (y_pred_arr == 1))[0][0])
non_churner_idx = int(np.where((y_test_arr == 0) & (y_pred_arr == 0))[0][0])

churn_prob    = model.predict_proba(X_test.iloc[[churner_idx]])[0][1]
no_churn_prob = model.predict_proba(X_test.iloc[[non_churner_idx]])[0][1]

print(f'Churner     - index: {churner_idx}, churn probability: {churn_prob:.2%}')
print(f'Non-churner - index: {non_churner_idx}, churn probability: {no_churn_prob:.2%}')

# Waterfall for churner
shap_exp_churner = shap.Explanation(
    values=shap_values[churner_idx],
    base_values=explainer.expected_value,
    data=X_test.iloc[churner_idx].values,
    feature_names=feature_names
)
plt.figure(figsize=(10, 7))
shap.waterfall_plot(shap_exp_churner, max_display=12, show=False)
plt.title(f'SHAP Waterfall: CHURNER  (P(churn)={churn_prob:.2%})', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{WORKSPACE}/shap_waterfall_churner.png', dpi=150, bbox_inches='tight')
plt.close()
print('Saved: shap_waterfall_churner.png')

# Waterfall for non-churner
shap_exp_non_churner = shap.Explanation(
    values=shap_values[non_churner_idx],
    base_values=explainer.expected_value,
    data=X_test.iloc[non_churner_idx].values,
    feature_names=feature_names
)
plt.figure(figsize=(10, 7))
shap.waterfall_plot(shap_exp_non_churner, max_display=12, show=False)
plt.title(f'SHAP Waterfall: NON-CHURNER  (P(churn)={no_churn_prob:.2%})', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{WORKSPACE}/shap_waterfall_non_churner.png', dpi=150, bbox_inches='tight')
plt.close()
print('Saved: shap_waterfall_non_churner.png')

Churner     - index: 9, churn probability: 65.91%
Non-churner - index: 0, churn probability: 0.40%


Saved: shap_waterfall_churner.png


Saved: shap_waterfall_non_churner.png


## Section 7: SHAP — Dependence Plot

Shows how a **single feature's value** affects model output across all customers. Reveals non-linear patterns the model learned.

In [10]:
top3_features = [feature_names[i] for i in top10_idx[:3]]
print(f'Dependence plots for: {top3_features}')

for feat in top3_features:
    plt.figure(figsize=(8, 5))
    shap.dependence_plot(
        feat, shap_values, X_test, feature_names=feature_names,
        interaction_index=None, show=False
    )
    plt.title(f'SHAP Dependence: {feat}', fontsize=12, fontweight='bold')
    plt.tight_layout()
    safe = feat.replace(' ', '_').replace('/', '_').replace('-', '_')
    plt.savefig(f'{WORKSPACE}/shap_dependence_{safe}.png', dpi=150, bbox_inches='tight')
    plt.close()
    print(f'  Saved: shap_dependence_{safe}.png')

Dependence plots for: ['Contract_Month-to-month', 'tenure', 'MonthlyCharges']
  Saved: shap_dependence_Contract_Month_to_month.png
  Saved: shap_dependence_tenure.png


  Saved: shap_dependence_MonthlyCharges.png


## Section 8: LIME — Create a LIME Explainer

**What is LIME?** LIME zooms in on one prediction, creates ~1,000 slightly-modified versions of that customer (perturbations), asks the model for predictions on all of them, then fits a **simple linear model** to those nearby predictions. The linear coefficients are the explanation.

| | SHAP | LIME |
|--|------|------|
| Accuracy | Exact | Approximate |
| Works on | Tree models | Any model |
| Uses | Game theory | Local linear model |

In [11]:
X_train_arr = X_train.values
X_test_arr  = X_test.values

# Declare all non-numeric-continuous features as categorical
# This prevents LIME from sampling truncnorm for binary 0/1 columns
continuous_feature_names = ['tenure', 'MonthlyCharges', 'TotalCharges']
categorical_feature_indices = [
    i for i, fn in enumerate(feature_names)
    if fn not in continuous_feature_names
]
print(f'Categorical features: {len(categorical_feature_indices)}')
print(f'Continuous features: {continuous_feature_names}')

# discretize_continuous=False avoids the LIME discretizer's zero-std bug
lime_explainer = lime.lime_tabular.LimeTabularExplainer(
    training_data=X_train_arr,
    feature_names=feature_names,
    class_names=['No Churn', 'Churn'],
    categorical_features=categorical_feature_indices,
    discretize_continuous=False,
    mode='classification',
    random_state=42
)
print('LIME explainer initialized.')
print(f'  Training data shape: {X_train_arr.shape}')


Categorical features: 37
Continuous features: ['tenure', 'MonthlyCharges', 'TotalCharges']
LIME explainer initialized.
  Training data shape: (5634, 40)


## Section 9: LIME — Explain Individual Predictions

In [12]:
# CHURNER
lime_exp_churner = lime_explainer.explain_instance(
    data_row=X_test_arr[churner_idx],
    predict_fn=model.predict_proba,
    num_features=12,
    num_samples=1000
)
print(f'LIME explanation - CHURNER (P(churn) = {churn_prob:.2%})')
print(f'\n{"Feature condition":<52} {"Weight":>8}  Direction')
print('-' * 75)
for feat, weight in lime_exp_churner.as_list(label=1):
    arrow = '-> CHURN' if weight > 0 else '-> STAY'
    print(f'  {feat[:50]:<50} {weight:>+8.4f}  {arrow}')

print()

# NON-CHURNER
lime_exp_non_churner = lime_explainer.explain_instance(
    data_row=X_test_arr[non_churner_idx],
    predict_fn=model.predict_proba,
    num_features=12,
    num_samples=1000
)
print(f'LIME explanation - NON-CHURNER (P(churn) = {no_churn_prob:.2%})')
print(f'\n{"Feature condition":<52} {"Weight":>8}  Direction')
print('-' * 75)
for feat, weight in lime_exp_non_churner.as_list(label=1):
    arrow = '-> CHURN' if weight > 0 else '-> STAY'
    print(f'  {feat[:50]:<50} {weight:>+8.4f}  {arrow}')

LIME explanation - CHURNER (P(churn) = 65.91%)

Feature condition                                      Weight  Direction
---------------------------------------------------------------------------
  Contract_Month-to-month=1                           +0.1950  -> CHURN
  tenure                                              -0.0917  -> STAY
  Contract_Two year=0                                 +0.0829  -> CHURN
  PaymentMethod_Electronic check=1                    +0.0620  -> CHURN
  InternetService_Fiber optic=0                       -0.0619  -> STAY
  OnlineSecurity_No=1                                 +0.0608  -> CHURN
  PhoneService=1                                      -0.0534  -> STAY
  MultipleLines_No=1                                  -0.0448  -> STAY
  Dependents=0                                        +0.0447  -> CHURN
  TechSupport_No=1                                    +0.0442  -> CHURN
  PaperlessBilling=1                                  +0.0394  -> CHURN
  StreamingTV_Y

## Section 10: LIME — Visualize Explanation

In [13]:
# CHURNER figure
fig = lime_exp_churner.as_pyplot_figure(label=1)
fig.set_size_inches(12, 6)
plt.title(f'LIME Explanation: CHURNER  (P(churn)={churn_prob:.2%})', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{WORKSPACE}/lime_churner.png', dpi=150, bbox_inches='tight')
plt.close()
print('Saved: lime_churner.png')

# NON-CHURNER figure
fig = lime_exp_non_churner.as_pyplot_figure(label=1)
fig.set_size_inches(12, 6)
plt.title(f'LIME Explanation: NON-CHURNER  (P(churn)={no_churn_prob:.2%})', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{WORKSPACE}/lime_non_churner.png', dpi=150, bbox_inches='tight')
plt.close()
print('Saved: lime_non_churner.png')

Saved: lime_churner.png


Saved: lime_non_churner.png


## Section 11: Side-by-Side Comparison of SHAP and LIME

Both methods answer: *'For this specific customer, which features mattered most?'*

- **Agreement** = both confirm the same driver = high confidence in the explanation
- **Disagreement** = LIME's local linear approximation couldn't fully capture the model's complexity

In [14]:
print('=' * 75)
print('  SHAP vs LIME: TOP FEATURES FOR THE CHURNER')
print('=' * 75)

shap_ranked = sorted(
    zip(feature_names, shap_values[churner_idx]),
    key=lambda x: abs(x[1]), reverse=True
)[:10]
lime_ranked = lime_exp_churner.as_list(label=1)[:10]

print(f'\n{"#":<3} {"SHAP Feature":<38} {"SHAP Val":>9}  |  {"LIME Feature Condition":<38} {"LIME Wt":>9}')
print('-' * 103)
for i in range(10):
    s_name = shap_ranked[i][0][:36] if i < len(shap_ranked) else ''
    s_val  = f'{shap_ranked[i][1]:+.4f}' if i < len(shap_ranked) else ''
    l_name = lime_ranked[i][0][:36] if i < len(lime_ranked) else ''
    l_val  = f'{lime_ranked[i][1]:+.4f}' if i < len(lime_ranked) else ''
    print(f'{i+1:<3} {s_name:<38} {s_val:>9}  |  {l_name:<38} {l_val:>9}')

# Agreement check
shap_top5 = {name for name, _ in shap_ranked[:5]}
lime_top5 = set()
for lime_feat, _ in lime_ranked[:5]:
    for fn in feature_names:
        if fn in lime_feat:
            lime_top5.add(fn)
            break

overlap = shap_top5 & lime_top5
print(f'\nTop-5 agreement: {len(overlap)}/5 features overlap')
print(f'  Shared: {sorted(overlap)}')

  SHAP vs LIME: TOP FEATURES FOR THE CHURNER

#   SHAP Feature                            SHAP Val  |  LIME Feature Condition                   LIME Wt
-------------------------------------------------------------------------------------------------------
1   Contract_Month-to-month                  +0.5075  |  Contract_Month-to-month=1                +0.1950
2   PaymentMethod_Electronic check           +0.2743  |  tenure                                   -0.0917
3   TotalCharges                             -0.2294  |  Contract_Two year=0                      +0.0829
4   InternetService_Fiber optic              -0.2175  |  PaymentMethod_Electronic check=1         +0.0620
5   MultipleLines_No                         -0.1792  |  InternetService_Fiber optic=0            -0.0619
6   OnlineSecurity_No                        +0.1506  |  OnlineSecurity_No=1                      +0.0608
7   InternetService_DSL                      -0.1481  |  PhoneService=1                           -0.0534
8 

In [15]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 6))
fig.suptitle(f'SHAP vs LIME - CHURNER  (P(churn)={churn_prob:.2%})', fontsize=14, fontweight='bold')

# SHAP panel
shap_feats = [x[0][:35] for x in shap_ranked[:10]]
shap_vals  = [x[1] for x in shap_ranked[:10]]
colors_shap = ['#d73027' if v > 0 else '#4575b4' for v in shap_vals]
ax1.barh(range(len(shap_feats)), shap_vals[::-1], color=colors_shap[::-1])
ax1.set_yticks(range(len(shap_feats)))
ax1.set_yticklabels(shap_feats[::-1], fontsize=9)
ax1.axvline(0, color='black', linewidth=0.8)
ax1.set_xlabel('SHAP Value', fontsize=11)
ax1.set_title('SHAP (exact, game-theory)', fontsize=12, fontweight='bold')

# LIME panel
lime_feats = [x[0][:35] for x in lime_ranked[:10]]
lime_vals  = [x[1] for x in lime_ranked[:10]]
colors_lime = ['#d73027' if v > 0 else '#4575b4' for v in lime_vals]
ax2.barh(range(len(lime_feats)), lime_vals[::-1], color=colors_lime[::-1])
ax2.set_yticks(range(len(lime_feats)))
ax2.set_yticklabels(lime_feats[::-1], fontsize=9)
ax2.axvline(0, color='black', linewidth=0.8)
ax2.set_xlabel('LIME Weight', fontsize=11)
ax2.set_title('LIME (local linear approximation)', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig(f'{WORKSPACE}/shap_vs_lime_comparison.png', dpi=150, bbox_inches='tight')
plt.close()
print('Saved: shap_vs_lime_comparison.png')

Saved: shap_vs_lime_comparison.png


## Final Summary & Interpretation

### Key Churn Drivers
| Feature | Effect | Insight |
|---------|--------|---------|
| `Contract_Month-to-month` | Strong churn | No lock-in |
| `tenure` (low) | High churn | New customers are risky |
| `MonthlyCharges` (high) | High churn | Value perception |
| `Contract_Two year` | Low churn | Loyal customers |
| `InternetService_Fiber optic` | Higher churn | Higher cost/expectations |
| `OnlineSecurity_No` | Higher churn | Less value from service |

### SHAP vs LIME Summary
| | SHAP | LIME |
|--|------|------|
| Method | Game theory (Shapley) | Local linear model |
| Accuracy | Exact (TreeExplainer) | Approximate |
| Scope | Global + Local | Local only |
| Works on | Tree models | Any model |
| Speed | Fast for trees | Slower (~1k samples) |

In [16]:
import os

print('=' * 60)
print('  FINAL SUMMARY')
print('=' * 60)
print(f'\nDataset:       {df.shape[0]:,} customers')
print(f'Features:      {len(feature_names)} (after encoding)')
print(f'Overall churn: {df["Churn"].mean():.1%}')
print(f'\nXGBoost Performance:')
print(f'  AUC-ROC:   {roc_auc_score(y_test, y_prob):.4f}')
print(f'  F1-Score:  {f1_score(y_test, y_pred):.4f}')
print(f'  Recall:    {recall_score(y_test, y_pred):.4f}')
print(f'  Precision: {precision_score(y_test, y_pred):.4f}')
print(f'\nTop 5 Global Churn Drivers (SHAP):')
for rank, idx in enumerate(top10_idx[:5], 1):
    print(f'  {rank}. {feature_names[idx]:<45} mean|SHAP|={mean_abs_shap[idx]:.4f}')
print(f'\nSaved output files:')
for fname in sorted(os.listdir(WORKSPACE)):
    if fname.endswith('.png') or fname.endswith('.ipynb'):
        size_kb = os.path.getsize(os.path.join(WORKSPACE, fname)) / 1024
        print(f'  {fname:<50} ({size_kb:.1f} KB)')
print('\nDone!')

  FINAL SUMMARY

Dataset:       7,043 customers
Features:      40 (after encoding)
Overall churn: 26.5%

XGBoost Performance:
  AUC-ROC:   0.8282
  F1-Score:  0.6117
  Recall:    0.7433
  Precision: 0.5196

Top 5 Global Churn Drivers (SHAP):
  1. Contract_Month-to-month                       mean|SHAP|=0.8254
  2. tenure                                        mean|SHAP|=0.6534
  3. MonthlyCharges                                mean|SHAP|=0.4270
  4. TotalCharges                                  mean|SHAP|=0.3297
  5. OnlineSecurity_No                             mean|SHAP|=0.2764

Saved output files:
  churn_explainability.ipynb                         (25.0 KB)
  confusion_matrix.png                               (32.3 KB)
  lime_churner.png                                   (59.6 KB)
  lime_non_churner.png                               (59.1 KB)
  shap_bar.png                                       (85.8 KB)
  shap_dependence_Contract_Month_to_month.png        (35.0 KB)
  shap_depende